In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/district_population_timeseries.csv")
df.head()

,year,district,population
0,2000,Gasabo,316528
1,2000,Musanze,301451
2,2001,Gasabo,328887
3,2001,Musanze,306421
4,2002,Gasabo,343031


In [4]:
import os
os.getcwd()

'/Users/denzelngabo/Desktop/UrbanPlus_API/notebooks'

In [5]:
import pandas as pd

df = pd.read_csv("../data/processed/district_population_timeseries.csv")
df.head()

,year,district,population
0,2000,Gasabo,316528
1,2000,Musanze,301451
2,2001,Gasabo,328887
3,2001,Musanze,306421
4,2002,Gasabo,343031


In [6]:
train = df[df ["year"] <= 2015]
test = df[df ["year"] > 2015]

print("Train rows:", len(train))
print("Test rows:", len(test))

Train rows: 32
Test rows: 10


In [7]:
from sklearn.linear_model import LinearRegression


gasabo_train = train[train["district"] == "Gasabo"]
gasabo_test = test[test["district"] == "Gasabo"]

X_train = gasabo_train[["year"]]
y_train = gasabo_train["population"]

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(gasabo_test[["year"]])

results = gasabo_test[["year", "population"]].copy()
results["predicted_population"] = predictions.round()
results["error_%"] = (((results["predicted_population"] - results["population"]) / results["population"]) * 100).round(1)
results

,year,population,predicted_population,error_%
32,2016,694635,647942.0,-6.7
34,2017,736235,670267.0,-9.0
36,2018,781024,692592.0,-11.3
38,2019,830744,714916.0,-13.9
40,2020,883424,737241.0,-16.5


In [8]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

results["rf_predicted"] = rf.predict(gasabo_test[["year"]]).round()
results["rf_error_%"] = ((results["rf_predicted"] - results["population"]) / results["population"] * 100).round(1)
results

,year,population,predicted_population,error_%,rf_predicted,rf_error_%
32,2016,694635,647942.0,-6.7,635850.0,-8.5
34,2017,736235,670267.0,-9.0,635850.0,-13.6
36,2018,781024,692592.0,-11.3,635850.0,-18.6
38,2019,830744,714916.0,-13.9,635850.0,-23.5
40,2020,883424,737241.0,-16.5,635850.0,-28.0


In [9]:
import numpy as np 

log_model = LinearRegression()
log_model.fit(X_train, np.log(y_train))

log_predictions = log_model.predict(gasabo_test[["year"]])
results["log_predicted"] = np.exp(log_predictions).round()
results["log_error_%"] = ((results["log_predicted"] - results["population"]) / results["population"] * 100).round(1)
results

,year,population,predicted_population,error_%,rf_predicted,rf_error_%,log_predicted,log_error_%
32,2016,694635,647942.0,-6.7,635850.0,-8.5,676148.0,-2.7
34,2017,736235,670267.0,-9.0,635850.0,-13.6,709944.0,-3.6
36,2018,781024,692592.0,-11.3,635850.0,-18.6,745430.0,-4.6
38,2019,830744,714916.0,-13.9,635850.0,-23.5,782689.0,-5.8
40,2020,883424,737241.0,-16.5,635850.0,-28.0,821810.0,-7.0


In [10]:
all_results = []

for name in ["Gasabo", "Musanze"]:
    # 1. Filter this district's practice and exam rows
    d_train = train[train["district"] == name]
    d_test = test[test["district"] == name]

    X_tr = d_train[["year"]]
    y_tr = d_train["population"]
    X_te = d_test[["year"]]

    # 2. Train both models on this district only
    lin = LinearRegression().fit(X_tr, y_tr)
    log_lin = LinearRegression().fit(X_tr, np.log(y_tr))

    # 3. Predict the exam years and measure errors
    out = d_test[["district", "year", "population"]].copy()
    out["linear_pred"] = lin.predict(X_te).round()
    out["log_pred"] = np.exp(log_lin.predict(X_te)).round()
    out["linear_err_%"] = ((out["linear_pred"] - out["population"]) / out["population"] * 100).round(1)
    out["log_err_%"] = ((out["log_pred"] - out["population"]) / out["population"] * 100).round(1)

    all_results.append(out)

# 4. Stack both districts into one table
comparison = pd.concat(all_results)
comparison

,district,year,population,linear_pred,log_pred,linear_err_%,log_err_%
32,Gasabo,2016,694635,647942.0,676148.0,-6.7,-2.7
34,Gasabo,2017,736235,670267.0,709944.0,-9.0,-3.6
36,Gasabo,2018,781024,692592.0,745430.0,-11.3,-4.6
38,Gasabo,2019,830744,714916.0,782689.0,-13.9,-5.8
40,Gasabo,2020,883424,737241.0,821810.0,-16.5,-7.0
33,Musanze,2016,402248,398013.0,400902.0,-1.1,-0.3
35,Musanze,2017,410173,404228.0,408187.0,-1.4,-0.5
37,Musanze,2018,418353,410443.0,415604.0,-1.9,-0.7
39,Musanze,2019,426621,416657.0,423157.0,-2.3,-0.8
41,Musanze,2020,435324,422872.0,430846.0,-2.9,-1.0


In [11]:
census_2022 = {"Gasabo": 879505, "Musanze": 476522}

rows = []
for name in ["Gasabo", "Musanze"]:
    d = df[df["district"] == name]
    model = LinearRegression().fit(d[["year"]], np.log(d["population"]))

    pred = np.exp(model.predict(pd.DataFrame({"year": [2022]})))[0]
    actual = census_2022[name]

    rows.append({
        "district": name,
        "predicted_2022": round(pred),
        "census_2022": actual,
        "error_%": round((pred - actual) / actual * 100, 1)
    })

final_test = pd.DataFrame(rows)
final_test

,district,predicted_2022,census_2022,error_%
0,Gasabo,946247,879505,7.6
1,Musanze,449337,476522,-5.7


In [12]:
comparison.to_csv("../data/processed/model_comparison_2016_2020.csv", index=False)
final_test.to_csv("../data/processed/census_2022_validation.csv", index=False)
print("Saved")

Saved
